In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import zscore
import time
from datetime import datetime
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==================== 설정 및 데이터 로드 ====================

# 한글 폰트 설정
# 사용하고 계신 OS에 맞춰 폰트를 설정해주세요. (예: AppleGothic, Malgun Gothic)
plt.rcParams['font.family'] = 'Malgun Gothic' 
plt.rcParams['axes.unicode_minus'] = False

# 결과 저장 폴더 생성
os.makedirs('추천결과', exist_ok=True)
os.makedirs('시각화자료', exist_ok=True)

print("🚀 이중 관점 맞춤형 카드 추천 시스템 시작")
print("   💡 취향 유사도 + 최대 절약 두 가지 관점 지원")
print("="*70)

# 데이터 로드
df_customer = pd.read_csv('./all_customer.csv')
df_card = pd.read_excel('./all_card.xlsx', engine='openpyxl')

print(f"✅ 데이터 로드 완료")
print(f"   - 전체 고객 수: {len(df_customer):,}명")
print(f"   - 전체 카드 수: {len(df_card):,}개")

# ==================== 군집 매핑 및 기본 설정 ====================

user_cluster_name_map = {
    0: '평균적인 쇼핑 & 가족러',
    1: '쇼핑 & 디지털 서비스 선호자',
    2: '쇼핑 & 생활 밀착형 사용자',
    3: '절약형 생활 밀착 사용자',
    4: '대중교통 & 문화 소비 사용자'
}

card_cluster_name_map = {
    0: '자동차 & 교통 특화 카드',
    1: '프리미엄 올라운더',
    2: '기본형카드',
    3: '가성비 올인원 카드',
    4: '글로벌 여행 특화 카드',
    5: '컬처 & 쇼핑 특화 카드'
}

feature_columns = [
    '대중교통_점수', '자가용_점수', '해외_점수', '여행_점수', '문화생활_점수', 
    '가족_점수', '쇼핑_점수', '생필품_점수', '납부(고정지출)_점수', '디지털결제_점수'
]

print(f"📊 고객 군집 분포:")
cluster_counts = df_customer['군집'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    print(f"   - 군집 {cluster_id} ({user_cluster_name_map[cluster_id]}): {count:,}명")

print("\n" + "="*70 + "\n")

# ==================== 절약액 계산을 위한 사전 준비 ====================

print("💰 절약액 계산 시스템 초기화...")

# 혜택 관련 컬럼 자동 탐지
all_benefit_cols = [col for col in df_card.columns if '(%)' in col or '포인트' in col]
print(f"   - 발견된 혜택 컬럼: {len(all_benefit_cols)}개")

# 소비금액-혜택 컬럼 매핑
col_mapping = {
    benefit_col.replace('(%)', '_이용금액').replace('포인트', '_이용금액'): benefit_col
    for benefit_col in all_benefit_cols
}

# 실제 존재하는 소비금액 컬럼만 선별
available_spending_cols = [
    spend_col for spend_col in col_mapping.keys() if spend_col in df_customer.columns
]
available_benefit_cols = [col_mapping[spend_col] for spend_col in available_spending_cols]

print(f"   - 매칭 가능한 소비-혜택 쌍: {len(available_spending_cols)}개")
print(f"   - 예시: {available_spending_cols[:3] if available_spending_cols else '없음'}")

# 카드 혜택 매트릭스 사전 계산 (전역 변수로 성능 최적화)
CARD_BENEFIT_MATRIX = df_card[available_benefit_cols].fillna(0) / 100

print("✅ 절약액 계산 시스템 준비 완료\n")

# ==================== Z-Score 기반 군집 매칭 (사전 계산) ====================

print("🔄 군집별 매칭 관계 분석 중...")

customer_cluster_mean = df_customer.groupby('군집')[feature_columns].mean()
card_cluster_mean = df_card.groupby('cluster')[feature_columns].mean()

customer_profile_z = customer_cluster_mean.apply(zscore)
card_profile_z = card_cluster_mean.apply(zscore)

similarity_matrix_z = cosine_similarity(customer_profile_z, card_profile_z)
similarity_df_z = pd.DataFrame(similarity_matrix_z, 
                               index=customer_cluster_mean.index, 
                               columns=card_cluster_mean.index)

best_matches_zscore = similarity_df_z.idxmax(axis=1)

print("✅ 군집 매칭 완료")
print("📋 고객군집 → 카드군집 매칭 결과:")
for user_cluster, card_cluster in best_matches_zscore.items():
    print(f"   - {user_cluster}번({user_cluster_name_map[user_cluster]}) → "
          f"{card_cluster}번({card_cluster_name_map[card_cluster]})")

print("\n" + "="*70 + "\n")

# ==================== 고도화된 추천 함수들 ====================

def filter_by_previous_optimized(df_customer, df_card, user_ids):
    """여러 사용자를 한번에 처리하는 최적화된 필터링 함수"""
    results = {}
    
    for user_id in user_ids:
        try:
            base_amount = df_customer.loc[df_customer['ID'] == user_id, '이용금액_신판_B0M'].iloc[0] * 1000
            performance_requirement = pd.to_numeric(df_card['직전 1개월 합계 실적 금액(만원)'], errors='coerce') * 10000
            filtered_df = df_card[(performance_requirement.isna()) | (base_amount >= performance_requirement)].copy()
            results[user_id] = filtered_df
        except (IndexError, KeyError):
            results[user_id] = df_card.copy()
    
    return results

def calculate_savings_batch(user_batch, filtered_cards_dict):
    """배치 단위로 절약액을 계산하는 최적화 함수"""
    if not available_spending_cols:
        return {}
    
    savings_results = {}
    
    for _, user_row in user_batch.iterrows():
        user_id = user_row['ID']
        try:
            # 사용자 소비 벡터 생성
            user_spending_vector = user_row[available_spending_cols] * 1000
            user_spending_vector.index = CARD_BENEFIT_MATRIX.columns
            
            # 해당 사용자가 선택 가능한 카드들의 절약액 계산
            available_cards = filtered_cards_dict[user_id]
            if available_cards.empty:
                savings_results[user_id] = pd.Series(dtype=float)
                continue
                
            # 선택 가능한 카드들만 필터링
            card_indices = available_cards.index
            card_benefit_subset = CARD_BENEFIT_MATRIX.loc[card_indices]
            
            # 내적 계산으로 절약액 산출
            estimated_savings = card_benefit_subset.dot(user_spending_vector)
            savings_results[user_id] = estimated_savings
            
        except Exception as e:
            savings_results[user_id] = pd.Series(dtype=float)
    
    return savings_results

def batch_recommendations(user_batch, df_customer, df_card, cluster_matches, method='similarity'):
    """★★ 핵심 개선: 취향 유사도 + 최대 절약 두 방식 완전 지원 ★★"""
    batch_results = []
    
    # 사용자 배치의 필터링된 카드를 미리 계산
    filtered_cards_dict = filter_by_previous_optimized(df_customer, df_card, user_batch['ID'].tolist())
    
    # 절약액 방식일 경우 미리 계산
    savings_dict = {}
    if method == 'savings':
        savings_dict = calculate_savings_batch(user_batch, filtered_cards_dict)
    
    for _, user_row in user_batch.iterrows():
        user_id = user_row['ID']
        user_cluster = user_row['군집']
        
        try:
            # 추천 카드 군집 찾기
            recommended_card_cluster = cluster_matches[user_cluster]
            candidate_cards = df_card[df_card['cluster'] == recommended_card_cluster]
            
            # 전월 실적 필터링
            filtered_candidates = filtered_cards_dict[user_id]
            filtered_candidates = filtered_candidates[filtered_candidates['cluster'] == recommended_card_cluster]
            
            if filtered_candidates.empty:
                result = {
                    'user_id': user_id,
                    'user_cluster': user_cluster,
                    'recommended_cluster': recommended_card_cluster,
                    'top_1_card': None,
                    'top_1_score': 0,
                    'available_cards': 0,
                    'method': method,
                    'score_type': 'similarity' if method == 'similarity' else 'savings_amount'
                }
            else:
                if method == 'similarity':
                    # ✅ 취향 유사도 기반 추천
                    user_vector = user_row[feature_columns].values.reshape(1, -1)
                    candidate_vectors = filtered_candidates[feature_columns].values
                    similarities = cosine_similarity(user_vector, candidate_vectors)[0]
                    
                    best_idx = similarities.argmax()
                    result = {
                        'user_id': user_id,
                        'user_cluster': user_cluster,
                        'recommended_cluster': recommended_card_cluster,
                        'top_1_card': filtered_candidates.iloc[best_idx]['카드명'],
                        'top_1_score': similarities[best_idx],
                        'available_cards': len(filtered_candidates),
                        'method': method,
                        'score_type': 'similarity'
                    }
                
                elif method == 'savings':
                    # ✅ 최대 절약 기반 추천 (완전 구현)
                    user_savings = savings_dict.get(user_id, pd.Series(dtype=float))
                    
                    if user_savings.empty:
                        result = {
                            'user_id': user_id,
                            'user_cluster': user_cluster,
                            'recommended_cluster': recommended_card_cluster,
                            'top_1_card': None,
                            'top_1_score': 0,
                            'available_cards': len(filtered_candidates),
                            'method': method,
                            'score_type': 'savings_amount'
                        }
                    else:
                        # 필터링된 후보 카드들의 절약액만 선택
                        candidate_savings = user_savings[filtered_candidates.index]
                        candidate_savings = candidate_savings.dropna()
                        
                        if candidate_savings.empty:
                            result = {
                                'user_id': user_id,
                                'user_cluster': user_cluster,
                                'recommended_cluster': recommended_card_cluster,
                                'top_1_card': None,
                                'top_1_score': 0,
                                'available_cards': len(filtered_candidates),
                                'method': method,
                                'score_type': 'savings_amount'
                            }
                        else:
                            best_card_idx = candidate_savings.idxmax()
                            best_card_name = df_card.loc[best_card_idx, '카드명']
                            best_savings = candidate_savings[best_card_idx]
                            
                            result = {
                                'user_id': user_id,
                                'user_cluster': user_cluster,
                                'recommended_cluster': recommended_card_cluster,
                                'top_1_card': best_card_name,
                                'top_1_score': best_savings,
                                'available_cards': len(filtered_candidates),
                                'method': method,
                                'score_type': 'savings_amount'
                            }
            
            batch_results.append(result)
            
        except Exception as e:
            # 에러 발생 시 기본값
            batch_results.append({
                'user_id': user_id,
                'user_cluster': user_cluster,
                'recommended_cluster': None,
                'top_1_card': None,
                'top_1_score': 0,
                'available_cards': 0,
                'method': method,
                'score_type': 'similarity' if method == 'similarity' else 'savings_amount',
                'error': str(e)
            })
    
    return pd.DataFrame(batch_results)

# ==================== 고급 시각화 함수 (두 관점 지원) ====================

def create_dual_visualizations(similarity_results, savings_results, df_customer, df_card, max_samples=20):
    """★★ 취향 유사도 vs 최대 절약 비교 시각화 생성 ★★"""
    print(f"🎨 두 관점 비교 시각화 {max_samples}개 생성 중...")
    
    # 두 방식 모두 성공한 사례만 선별
    sim_success = similarity_results[similarity_results['top_1_card'].notna()]
    sav_success = savings_results[savings_results['top_1_card'].notna()]
    
    # 공통 성공 사례 찾기
    common_users = set(sim_success['user_id']) & set(sav_success['user_id'])
    if not common_users:
        print("⚠️  두 방식 모두 성공한 사례가 없어 비교 시각화를 건너뜁니다.")
        return []
    
    # 군집별로 균등하게 선택
    viz_cases = []
    common_df = pd.DataFrame({'user_id': list(common_users)})
    
    # 사용자들의 군집 정보 추가
    common_df = common_df.merge(df_customer[['ID', '군집']], left_on='user_id', right_on='ID')
    
    for cluster in common_df['군집'].unique():
        cluster_cases = common_df[common_df['군집'] == cluster]
        n_samples = min(4, len(cluster_cases))  # 군집별 최대 4개
        selected = cluster_cases.head(n_samples)
        viz_cases.extend(selected['user_id'].tolist())
    
    # 최대 개수 제한
    viz_cases = viz_cases[:max_samples]
    
    viz_files = []
    
    for i, user_id in enumerate(tqdm(viz_cases, desc="비교 시각화 생성")):
        try:
            # 두 방식의 추천 결과 가져오기
            sim_result = sim_success[sim_success['user_id'] == user_id].iloc[0]
            sav_result = sav_success[sav_success['user_id'] == user_id].iloc[0]
            
            sim_card = sim_result['top_1_card']
            sav_card = sav_result['top_1_card']
            
            # 사용자 프로필
            user_vector = df_customer[df_customer['ID'] == user_id][feature_columns].iloc[0].values
            
            # 두 카드의 프로필
            sim_card_vector = df_card[df_card['카드명'] == sim_card][feature_columns].iloc[0].values
            sav_card_vector = df_card[df_card['카드명'] == sav_card][feature_columns].iloc[0].values
            
            # 레이더 차트 생성
            categories = feature_columns
            N = len(categories)
            angles = [n / float(N) * 2 * np.pi for n in range(N)]
            angles += angles[:1]
            
            user_plot = user_vector.tolist() + [user_vector[0]]
            sim_plot = sim_card_vector.tolist() + [sim_card_vector[0]]
            sav_plot = sav_card_vector.tolist() + [sav_card_vector[0]]
            
            plt.figure(figsize=(14, 10))
            ax = plt.subplot(111, polar=True)
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(categories, fontsize=10)
            plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="grey", size=8)
            plt.ylim(0, 5.5)
            
            # 세 개의 프로필 플롯
            ax.plot(angles, user_plot, 'o-', linewidth=3, color='navy', 
                   label=f'사용자 (군집{sim_result["user_cluster"]})')
            ax.fill(angles, user_plot, 'navy', alpha=0.05)
            
            ax.plot(angles, sim_plot, 's-', linewidth=2, color='cornflowerblue', 
                   label=f'취향추천: {sim_card[:20]}...')
            ax.fill(angles, sim_plot, 'cornflowerblue', alpha=0.1)
            
            ax.plot(angles, sav_plot, '^-', linewidth=2, color='darkorange', 
                   label=f'절약추천: {sav_card[:20]}...')
            ax.fill(angles, sav_plot, 'darkorange', alpha=0.1)
            
            plt.legend(loc='upper right', bbox_to_anchor=(1.4, 1.0), fontsize=10)
            
            # 제목에 성과 지표 포함
            sim_score = f"유사도 {sim_result['top_1_score']:.3f}"
            sav_score = f"절약 {int(sav_result['top_1_score']):,}원/월"
            
            plt.title(f"비교분석 {i+1}: {sim_score} vs {sav_score}", 
                     size=16, y=1.15)
            
            filename = f'시각화자료/비교분석_{i+1:02d}_군집{sim_result["user_cluster"]}.png'
            plt.savefig(filename, bbox_inches='tight', dpi=150)
            plt.close()
            
            viz_files.append(filename)
            
        except Exception as e:
            print(f"⚠️  사례 {i+1} 시각화 실패: {e}")
    
    print(f"✅ {len(viz_files)}개 비교 시각화 파일 생성 완료")
    return viz_files

# ==================== 고급 성과 분석 함수들 ====================

def analyze_dual_performance(similarity_results, savings_results):
    """★★ 두 관점 성과 비교 분석 ★★"""
    
    print("\n" + "="*70)
    print("📊 취향 유사도 vs 최대 절약 성과 비교 분석")
    print("="*70)
    
    # 기본 통계
    sim_success_rate = (similarity_results['top_1_card'].notna().sum() / len(similarity_results)) * 100
    sav_success_rate = (savings_results['top_1_card'].notna().sum() / len(savings_results)) * 100
    
    sim_avg_score = similarity_results['top_1_score'].mean()
    sav_avg_score = savings_results['top_1_score'].mean()
    
    print(f"📈 전체 성과 비교:")
    print(f"   - 취향 유사도: 성공률 {sim_success_rate:.1f}%, 평균 유사도 {sim_avg_score:.3f}")
    print(f"   - 최대 절약:   성공률 {sav_success_rate:.1f}%, 평균 절약액 {sav_avg_score:,.0f}원/월")
    
    # 군집별 성과 비교
    sim_cluster = similarity_results.groupby('user_cluster').agg({
        'user_id': 'count',
        'top_1_card': lambda x: x.notna().sum(),
        'top_1_score': 'mean'
    }).round(3)
    sim_cluster.columns = ['총고객수', '성공추천수', '평균유사도']
    sim_cluster['성공률(%)'] = (sim_cluster['성공추천수'] / sim_cluster['총고객수'] * 100).round(1)
    
    sav_cluster = savings_results.groupby('user_cluster').agg({
        'user_id': 'count', 
        'top_1_card': lambda x: x.notna().sum(),
        'top_1_score': 'mean'
    }).round(0)
    sav_cluster.columns = ['총고객수', '성공추천수', '평균절약액']
    sav_cluster['성공률(%)'] = (sav_cluster['성공추천수'] / sav_cluster['총고객수'] * 100).round(1)
    
    print(f"\n🎯 군집별 취향 유사도 성과:")
    sim_cluster.index = sim_cluster.index.map(user_cluster_name_map)
    print(sim_cluster)
    
    print(f"\n💰 군집별 최대 절약 성과:")
    sav_cluster.index = sav_cluster.index.map(user_cluster_name_map)
    sav_cluster['평균절약액'] = sav_cluster['평균절약액'].apply(lambda x: f"{int(x):,}원")
    print(sav_cluster)
    
    # 추천 카드 중복도 분석
    sim_cards = set(similarity_results[similarity_results['top_1_card'].notna()]['top_1_card'])
    sav_cards = set(savings_results[savings_results['top_1_card'].notna()]['top_1_card'])
    
    overlap_cards = sim_cards & sav_cards
    overlap_ratio = len(overlap_cards) / len(sim_cards | sav_cards) * 100 if sim_cards | sav_cards else 0
    
    print(f"\n🔄 추천 결과 중복 분석:")
    print(f"   - 취향추천 고유카드: {len(sim_cards - sav_cards):,}개")
    print(f"   - 절약추천 고유카드: {len(sav_cards - sim_cards):,}개") 
    print(f"   - 공통 추천카드: {len(overlap_cards):,}개")
    print(f"   - 중복률: {overlap_ratio:.1f}%")
    
    return {
        'similarity_cluster_perf': sim_cluster,
        'savings_cluster_perf': sav_cluster,
        'overlap_analysis': {
            'sim_unique': len(sim_cards - sav_cards),
            'sav_unique': len(sav_cards - sim_cards),
            'common': len(overlap_cards),
            'overlap_ratio': overlap_ratio
        }
    }

# ==================== 단계별 실행 함수들 (두 관점 지원) ====================

def stage1_dual_prototype(n_per_cluster=100):
    """1단계: 두 관점으로 프로토타입 검증"""
    print("🔍 1단계: 이중 관점 프로토타입 검증 시작")
    print(f"   - 군집별 최대 {n_per_cluster}명씩 샘플링")
    print("   - 취향 유사도 + 최대 절약 두 방식 동시 실행")
    
    # 군집별 샘플링
    sample_customers = []
    for cluster_id in df_customer['군집'].unique():
        cluster_data = df_customer[df_customer['군집'] == cluster_id]
        n_samples = min(n_per_cluster, len(cluster_data))
        sampled = cluster_data.sample(n=n_samples, random_state=42)
        sample_customers.append(sampled)
    
    sample_df = pd.concat(sample_customers, ignore_index=True)
    print(f"✅ 총 {len(sample_df):,}명 샘플링 완료")
    
    # 배치 처리로 두 방식 동시 추천
    start_time = time.time()
    batch_size = 100
    
    sim_results = []
    sav_results = []
    
    print("🔄 두 관점 동시 추천 처리 중...")
    for i in tqdm(range(0, len(sample_df), batch_size), desc="1단계 이중 추천"):
        batch = sample_df.iloc[i:i+batch_size]
        
        # 취향 유사도 방식
        sim_batch = batch_recommendations(batch, df_customer, df_card, best_matches_zscore, 'similarity')
        sim_results.append(sim_batch)
        
        # 최대 절약 방식
        sav_batch = batch_recommendations(batch, df_customer, df_card, best_matches_zscore, 'savings')
        sav_results.append(sav_batch)
    
    similarity_results = pd.concat(sim_results, ignore_index=True)
    savings_results = pd.concat(sav_results, ignore_index=True)
    
    processing_time = time.time() - start_time
    print(f"⏱️  총 처리 시간: {processing_time:.2f}초")
    
    # 성과 분석
    analysis_results = analyze_dual_performance(similarity_results, savings_results)
    
    # 결과 저장
    similarity_results.to_csv('추천결과/1단계_취향유사도_결과.csv', index=False, encoding='utf-8-sig')
    savings_results.to_csv('추천결과/1단계_최대절약_결과.csv', index=False, encoding='utf-8-sig')
    analysis_results['similarity_cluster_perf'].to_csv('추천결과/1단계_취향_군집성과.csv', encoding='utf-8-sig')
    analysis_results['savings_cluster_perf'].to_csv('추천결과/1단계_절약_군집성과.csv', encoding='utf-8-sig')
    
    # 비교 시각화 생성
    viz_files = create_dual_visualizations(similarity_results, savings_results, df_customer, df_card, 12)
    
    print(f"\n✅ 1단계 완료 - 이중 관점 분석 결과 저장됨")
    return similarity_results, savings_results, analysis_results

def stage2_dual_executive(sample_ratio=0.1):
    """2단계: 두 관점으로 임원진 발표용 결과 생성"""
    print(f"🎯 2단계: 이중 관점 {sample_ratio*100}% 샘플 임원진 발표용 분석")
    
    # 층화표집 샘플링
    sample_customers = []
    for cluster_id in df_customer['군집'].unique():
        cluster_data = df_customer[df_customer['군집'] == cluster_id]
        n_samples = int(len(cluster_data) * sample_ratio)
        if n_samples > 0:
            sampled = cluster_data.sample(n=n_samples, random_state=42)
            sample_customers.append(sampled)
    
    sample_df = pd.concat(sample_customers, ignore_index=True)
    print(f"✅ 총 {len(sample_df):,}명 샘플링 완료")
    
    # 대량 배치 처리
    start_time = time.time()
    batch_size = 500
    
    sim_results = []
    sav_results = []
    
    print("🔄 대량 이중 추천 처리 중...")
    for i in tqdm(range(0, len(sample_df), batch_size), desc="2단계 이중 추천"):
        batch = sample_df.iloc[i:i+batch_size]
        
        sim_batch = batch_recommendations(batch, df_customer, df_card, best_matches_zscore, 'similarity')
        sim_results.append(sim_batch)
        
        sav_batch = batch_recommendations(batch, df_customer, df_card, best_matches_zscore, 'savings')
        sav_results.append(sav_batch)
    
    similarity_results = pd.concat(sim_results, ignore_index=True)
    savings_results = pd.concat(sav_results, ignore_index=True)
    
    processing_time = time.time() - start_time
    
    # 종합 성과 분석
    analysis_results = analyze_dual_performance(similarity_results, savings_results)
    
    # Executive Summary 생성
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    
    with open(f'추천결과/2단계_이중관점_경영진요약_{timestamp}.txt', 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("이중 관점 맞춤형 카드 추천 시스템 - 경영진 요약 보고서\n")
        f.write("="*60 + "\n\n")
        f.write(f"📅 분석 일시: {datetime.now().strftime('%Y년 %m월 %d일 %H시 %M분')}\n")
        f.write(f"👥 분석 대상: 전체 고객의 {sample_ratio*100}% ({len(similarity_results):,}명)\n")
        f.write(f"⏱️  처리 시간: {processing_time:.1f}초\n\n")
        
        sim_success_rate = (similarity_results['top_1_card'].notna().sum() / len(similarity_results)) * 100
        sav_success_rate = (savings_results['top_1_card'].notna().sum() / len(savings_results)) * 100
        sav_avg_savings = savings_results['top_1_score'].mean()
        
        f.write("📊 핵심 성과 지표:\n")
        f.write(f"- 취향 유사도 추천: 성공률 {sim_success_rate:.1f}%\n")
        f.write(f"- 최대 절약 추천:   성공률 {sav_success_rate:.1f}%, 평균 {sav_avg_savings:,.0f}원/월\n")
        f.write(f"- 추천 결과 중복률: {analysis_results['overlap_analysis']['overlap_ratio']:.1f}%\n\n")
        
        f.write("💡 전략적 인사이트:\n")
        f.write("- 두 추천 방식의 상호 보완적 활용으로 고객 만족도 극대화 가능\n")
        f.write("- 취향 중심 고객과 실용성 중심 고객의 차별화된 접근 필요\n")
        
        if sav_avg_savings > 0:
            annual_savings = sav_avg_savings * 12
            f.write(f"- 연간 예상 고객 절약액: {annual_savings:,.0f}원 (마케팅 포인트 활용 가능)\n")
    
    # 결과 저장
    similarity_results.to_csv(f'추천결과/2단계_취향유사도_{timestamp}.csv', index=False, encoding='utf-8-sig')
    savings_results.to_csv(f'추천결과/2단계_최대절약_{timestamp}.csv', index=False, encoding='utf-8-sig')
    
    # 대표 시각화 생성
    viz_files = create_dual_visualizations(similarity_results, savings_results, df_customer, df_card, 25)
    
    print(f"\n✅ 2단계 완료 - 이중 관점 임원진 발표 자료 준비됨")
    print(f"📁 모든 결과 파일이 '추천결과/' 폴더에 저장되었습니다.")
    print(f"🎨 {len(viz_files)}개 비교 시각화가 '시각화자료/' 폴더에 저장되었습니다.")
    
    return similarity_results, savings_results, analysis_results

def stage3_full_dual_deployment():
    """3단계: 전체 고객 대상 이중 추천"""
    print("🚀 3단계: 전체 고객 대상 이중 추천 시작")
    print("⚠️  주의: 이는 실제 서비스 배포를 위한 대량 처리입니다.")
    
    confirm = input("전체 18만+ 고객에 대한 이중 추천을 진행하시겠습니까? (y/N): ")
    if confirm.lower() != 'y':
        print("🛑 3단계 처리를 취소했습니다.")
        return None
    
    print("🔄 전체 고객 대상 이중 추천 처리 시작...")
    print("⏸️  3단계는 1,2단계 성과 검증 후 구현 예정입니다.")
    return None

# ==================== 메인 실행부 ====================

def main():
    print("🎯 이중 관점 맞춤형 카드 추천 시스템")
    print("   💡 취향 유사도: 고객의 선호 패턴과 유사한 카드 추천")
    print("   💰 최대 절약: 고객의 소비 패턴으로 가장 많이 절약되는 카드 추천")
    print("")
    print("1️⃣  1단계: 프로토타입 검증 (군집별 대표 고객 ~500명)")
    print("2️⃣  2단계: 임원진 발표용 (10% 샘플 ~18,000명)")  
    print("3️⃣  3단계: 전체 배포 (전체 고객 180,000+명)")
    
    while True:
        choice = input("\n실행할 단계를 선택하세요 (1/2/3/q): ").strip()
        
        if choice == '1':
            print("\n" + "="*70)
            sim_results, sav_results, analysis = stage1_dual_prototype()
            
        elif choice == '2':
            print("\n" + "="*70)
            sim_results, sav_results, analysis = stage2_dual_executive()
            
        elif choice == '3':
            print("\n" + "="*70)
            stage3_full_dual_deployment()
            
        elif choice.lower() == 'q':
            print("👋 이중 관점 추천 시스템을 종료합니다.")
            break
            
        else:
            print("❌ 잘못된 선택입니다. 1, 2, 3, 또는 q를 입력해주세요.")

# 실행
if __name__ == "__main__":
    main()


🚀 이중 관점 맞춤형 카드 추천 시스템 시작
   💡 취향 유사도 + 최대 절약 두 가지 관점 지원
✅ 데이터 로드 완료
   - 전체 고객 수: 187,552명
   - 전체 카드 수: 213개
📊 고객 군집 분포:
   - 군집 0 (평균적인 쇼핑 & 가족러): 9,330명
   - 군집 1 (쇼핑 & 디지털 서비스 선호자): 33,864명
   - 군집 2 (쇼핑 & 생활 밀착형 사용자): 79,265명
   - 군집 3 (절약형 생활 밀착 사용자): 21,899명
   - 군집 4 (대중교통 & 문화 소비 사용자): 43,194명


💰 절약액 계산 시스템 초기화...
   - 발견된 혜택 컬럼: 25개
   - 매칭 가능한 소비-혜택 쌍: 4개
   - 예시: ['쇼핑_온라인_이용금액', '쇼핑_백화점_이용금액', '쇼핑_아울렛_이용금액']
✅ 절약액 계산 시스템 준비 완료

🔄 군집별 매칭 관계 분석 중...
✅ 군집 매칭 완료
📋 고객군집 → 카드군집 매칭 결과:
   - 0번(평균적인 쇼핑 & 가족러) → 2번(기본형카드)
   - 1번(쇼핑 & 디지털 서비스 선호자) → 2번(기본형카드)
   - 2번(쇼핑 & 생활 밀착형 사용자) → 1번(프리미엄 올라운더)
   - 3번(절약형 생활 밀착 사용자) → 0번(자동차 & 교통 특화 카드)
   - 4번(대중교통 & 문화 소비 사용자) → 3번(가성비 올인원 카드)


🎯 이중 관점 맞춤형 카드 추천 시스템
   💡 취향 유사도: 고객의 선호 패턴과 유사한 카드 추천
   💰 최대 절약: 고객의 소비 패턴으로 가장 많이 절약되는 카드 추천

1️⃣  1단계: 프로토타입 검증 (군집별 대표 고객 ~500명)
2️⃣  2단계: 임원진 발표용 (10% 샘플 ~18,000명)
3️⃣  3단계: 전체 배포 (전체 고객 180,000+명)

🔍 1단계: 이중 관점 프로토타입 검증 시작
   - 군집별 최대 100명씩 샘플링
   - 취향 유사도 + 최대 절약 두 방식 동시 실행
✅ 총 500명 샘플링 완료


1단계 이중 추천: 100%|██████████| 5/5 [00:06<00:00,  1.30s/it]


⏱️  총 처리 시간: 6.51초

📊 취향 유사도 vs 최대 절약 성과 비교 분석
📈 전체 성과 비교:
   - 취향 유사도: 성공률 100.0%, 평균 유사도 0.836
   - 최대 절약:   성공률 100.0%, 평균 절약액 109,785원/월

🎯 군집별 취향 유사도 성과:
                  총고객수  성공추천수  평균유사도  성공률(%)
user_cluster                                
평균적인 쇼핑 & 가족러      100    100  0.836   100.0
쇼핑 & 디지털 서비스 선호자   100    100  0.795   100.0
쇼핑 & 생활 밀착형 사용자    100    100  0.901   100.0
절약형 생활 밀착 사용자      100    100  0.721   100.0
대중교통 & 문화 소비 사용자   100    100  0.929   100.0

💰 군집별 최대 절약 성과:
                  총고객수  성공추천수     평균절약액  성공률(%)
user_cluster                                   
평균적인 쇼핑 & 가족러      100    100   13,122원   100.0
쇼핑 & 디지털 서비스 선호자   100    100   14,771원   100.0
쇼핑 & 생활 밀착형 사용자    100    100  474,688원   100.0
절약형 생활 밀착 사용자      100    100    9,946원   100.0
대중교통 & 문화 소비 사용자   100    100   36,397원   100.0

🔄 추천 결과 중복 분석:
   - 취향추천 고유카드: 54개
   - 절약추천 고유카드: 7개
   - 공통 추천카드: 7개
   - 중복률: 10.3%
🎨 두 관점 비교 시각화 12개 생성 중...


비교 시각화 생성: 100%|██████████| 12/12 [00:03<00:00,  3.58it/s]


✅ 12개 비교 시각화 파일 생성 완료

✅ 1단계 완료 - 이중 관점 분석 결과 저장됨

🔍 1단계: 이중 관점 프로토타입 검증 시작
   - 군집별 최대 100명씩 샘플링
   - 취향 유사도 + 최대 절약 두 방식 동시 실행
✅ 총 500명 샘플링 완료
🔄 두 관점 동시 추천 처리 중...


...

✅ 12개 비교 시각화 파일 생성 완료

✅ 1단계 완료 - 이중 관점 분석 결과 저장됨

🔍 1단계: 이중 관점 프로토타입 검증 시작
   - 군집별 최대 100명씩 샘플링
   - 취향 유사도 + 최대 절약 두 방식 동시 실행
✅ 총 500명 샘플링 완료
🔄 두 관점 동시 추천 처리 중...


1단계 이중 추천: 100%|██████████| 5/5 [00:07<00:00,  1.48s/it]


⏱️  총 처리 시간: 7.38초

📊 취향 유사도 vs 최대 절약 성과 비교 분석
📈 전체 성과 비교:
   - 취향 유사도: 성공률 100.0%, 평균 유사도 0.836
   - 최대 절약:   성공률 100.0%, 평균 절약액 109,785원/월

🎯 군집별 취향 유사도 성과:
                  총고객수  성공추천수  평균유사도  성공률(%)
user_cluster                                
평균적인 쇼핑 & 가족러      100    100  0.836   100.0
쇼핑 & 디지털 서비스 선호자   100    100  0.795   100.0
쇼핑 & 생활 밀착형 사용자    100    100  0.901   100.0
절약형 생활 밀착 사용자      100    100  0.721   100.0
대중교통 & 문화 소비 사용자   100    100  0.929   100.0

💰 군집별 최대 절약 성과:
                  총고객수  성공추천수     평균절약액  성공률(%)
user_cluster                                   
평균적인 쇼핑 & 가족러      100    100   13,122원   100.0
쇼핑 & 디지털 서비스 선호자   100    100   14,771원   100.0
쇼핑 & 생활 밀착형 사용자    100    100  474,688원   100.0
절약형 생활 밀착 사용자      100    100    9,946원   100.0
대중교통 & 문화 소비 사용자   100    100   36,397원   100.0

🔄 추천 결과 중복 분석:
   - 취향추천 고유카드: 54개
   - 절약추천 고유카드: 7개
   - 공통 추천카드: 7개
   - 중복률: 10.3%
🎨 두 관점 비교 시각화 12개 생성 중...


비교 시각화 생성: 100%|██████████| 12/12 [00:03<00:00,  3.92it/s]


✅ 12개 비교 시각화 파일 생성 완료

✅ 1단계 완료 - 이중 관점 분석 결과 저장됨
👋 이중 관점 추천 시스템을 종료합니다.
